In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)  # for reproducibility

def generate_engine_data(n_samples=5000):
    # Basic feature distributions
    mileage = np.random.uniform(10000, 300000, n_samples)                 # km
    engine_temp = np.random.normal(90, 10, n_samples)                     # °C
    oil_pressure = np.random.normal(3.0, 0.7, n_samples)                  # bar
    rpm = np.random.uniform(600, 4500, n_samples)                         # rpm
    error_code_count = np.random.poisson(1.0, n_samples)                  # count
    time_since_last_service = np.random.uniform(0, 700, n_samples)        # days
    avg_trip_duration = np.random.uniform(5, 90, n_samples)               # minutes
    city_drive_ratio = np.random.uniform(0, 1, n_samples)                 # 0–1

    # Clip some values to realistic ranges
    engine_temp = np.clip(engine_temp, 60, 130)
    oil_pressure = np.clip(oil_pressure, 0.5, 6.0)
    error_code_count = np.clip(error_code_count, 0, 10)

    # Make a “risk score” combining features
    risk_score = (
        (mileage - 100000) / 100000 +                         # higher mileage = more risk
        (engine_temp - 90) / 20 +                             # hotter engine = more risk
        (3.0 - oil_pressure) / 1.5 +                          # lower oil pressure = more risk
        (error_code_count) * 0.5 +                            # more codes = more risk
        (time_since_last_service - 365) / 365 +               # long time since service = more risk
        city_drive_ratio * 0.5                                # more city driving = slightly more risk
    )

    # Convert risk_score to probability of failure via logistic function
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))
    
    # Shift and scale risk_score to get reasonable probabilities
    prob_failure = sigmoid(risk_score - 0.5)

    # Sample the label: will_fail_soon (1) or not (0)
    will_fail_soon = np.random.binomial(1, prob_failure)

    data = pd.DataFrame({
        "mileage": mileage,
        "engine_temp": engine_temp,
        "oil_pressure": oil_pressure,
        "rpm": rpm,
        "error_code_count": error_code_count,
        "time_since_last_service": time_since_last_service,
        "avg_trip_duration": avg_trip_duration,
        "city_drive_ratio": city_drive_ratio,
        "will_fail_soon": will_fail_soon
    })

    return data

# Generate and inspect data
data = generate_engine_data(5000)
print(data.head())
print("\nClass balance:")
print(data["will_fail_soon"].value_counts(normalize=True))

# Save to CSV so you can reuse it
data.to_csv("engine_data_synthetic.csv", index=False)
print("\nSaved to engine_data_synthetic.csv")


         mileage  engine_temp  oil_pressure          rpm  error_code_count  \
0  118616.634466    84.030257      2.898255   839.485674                 0   
1  285707.148859    66.096956      3.243178  4183.845498                 0   
2  222278.243125    85.877793      2.308398   916.192173                 0   
3  183610.960417    99.134737      2.896965  4165.502368                 2   
4   55245.405728    95.376299      2.860656  2514.148273                 3   

   time_since_last_service  avg_trip_duration  city_drive_ratio  \
0               691.796573          30.927273          0.046765   
1               665.383151          59.329520          0.122111   
2               514.707675          54.242596          0.296191   
3               679.750351          78.846764          0.259113   
4                73.943704          72.254632          0.728327   

   will_fail_soon  
0               0  
1               1  
2               1  
3               1  
4               0  

Class b

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib  # to save the model

# 1. Load data
data = pd.read_csv("engine_data_synthetic.csv")

# 2. Split into features and label
X = data.drop("will_fail_soon", axis=1)
y = data["will_fail_soon"]

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"  # helps if classes are imbalanced
)

model.fit(X_train, y_train)

# 5. Evaluation
y_pred = model.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

# 6. Save model
joblib.dump(model, "engine_failure_predictor.joblib")
print("\nModel saved to engine_failure_predictor.joblib")


Confusion matrix:
[[173 198]
 [ 91 538]]

Classification report:
              precision    recall  f1-score   support

           0       0.66      0.47      0.54       371
           1       0.73      0.86      0.79       629

    accuracy                           0.71      1000
   macro avg       0.69      0.66      0.67      1000
weighted avg       0.70      0.71      0.70      1000


Model saved to engine_failure_predictor.joblib


In [3]:
def predict_engine_failure(model, mileage, engine_temp, oil_pressure,
                           rpm, error_code_count, time_since_last_service,
                           avg_trip_duration, city_drive_ratio, threshold=0.7):
    # Put inputs in a DataFrame (same order as training features)
    input_df = pd.DataFrame([{
        "mileage": mileage,
        "engine_temp": engine_temp,
        "oil_pressure": oil_pressure,
        "rpm": rpm,
        "error_code_count": error_code_count,
        "time_since_last_service": time_since_last_service,
        "avg_trip_duration": avg_trip_duration,
        "city_drive_ratio": city_drive_ratio
    }])

    prob_failure = model.predict_proba(input_df)[0][1]

    status = "OK"
    msg = "✅ Engine is unlikely to fail in the near future."
    if prob_failure >= threshold:
        status = "RISK"
        msg = "⚠️ Engine is at higher risk of failure soon. Recommend inspection."

    return prob_failure, status, msg

# Example usage:
loaded_model = joblib.load("engine_failure_predictor.joblib")

prob, status, msg = predict_engine_failure(
    loaded_model,
    mileage=220000,
    engine_temp=105,
    oil_pressure=1.8,
    rpm=2800,
    error_code_count=4,
    time_since_last_service=500,
    avg_trip_duration=20,
    city_drive_ratio=0.8
)

print("Probability of failure:", prob)
print("Status:", status)
print("Message:", msg)


Probability of failure: 0.9566666666666667
Status: RISK
Message: ⚠️ Engine is at higher risk of failure soon. Recommend inspection.


In [4]:
pip install numpy pandas scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import tkinter as tk
from tkinter import ttk, messagebox

# =========================
# 1. DATA GENERATION + MODEL
# =========================

def generate_engine_data(n_samples=5000):
    np.random.seed(42)

    mileage = np.random.uniform(10000, 300000, n_samples)                 # km
    engine_temp = np.random.normal(90, 10, n_samples)                     # °C
    oil_pressure = np.random.normal(3.0, 0.7, n_samples)                  # bar
    rpm = np.random.uniform(600, 4500, n_samples)                         # rpm
    error_code_count = np.random.poisson(1.0, n_samples)                  # count
    time_since_last_service = np.random.uniform(0, 700, n_samples)        # days
    avg_trip_duration = np.random.uniform(5, 90, n_samples)               # minutes
    city_drive_ratio = np.random.uniform(0, 1, n_samples)                 # 0–1

    engine_temp = np.clip(engine_temp, 60, 130)
    oil_pressure = np.clip(oil_pressure, 0.5, 6.0)
    error_code_count = np.clip(error_code_count, 0, 10)

    risk_score = (
        (mileage - 100000) / 100000 +
        (engine_temp - 90) / 20 +
        (3.0 - oil_pressure) / 1.5 +
        (error_code_count) * 0.5 +
        (time_since_last_service - 365) / 365 +
        city_drive_ratio * 0.5
    )

    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    prob_failure = sigmoid(risk_score - 0.5)
    will_fail_soon = np.random.binomial(1, prob_failure)

    data = pd.DataFrame({
        "mileage": mileage,
        "engine_temp": engine_temp,
        "oil_pressure": oil_pressure,
        "rpm": rpm,
        "error_code_count": error_code_count,
        "time_since_last_service": time_since_last_service,
        "avg_trip_duration": avg_trip_duration,
        "city_drive_ratio": city_drive_ratio,
        "will_fail_soon": will_fail_soon
    })

    return data

print("Generating synthetic engine data...")
data = generate_engine_data(5000)

X = data.drop("will_fail_soon", axis=1)
y = data["will_fail_soon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training RandomForest model...")
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
model.fit(X_train, y_train)
print("Model training complete.")


def predict_engine_failure(
    mileage, engine_temp, oil_pressure, rpm,
    error_code_count, time_since_last_service,
    avg_trip_duration, city_drive_ratio,
    threshold=0.7
):
    input_df = pd.DataFrame([{
        "mileage": mileage,
        "engine_temp": engine_temp,
        "oil_pressure": oil_pressure,
        "rpm": rpm,
        "error_code_count": error_code_count,
        "time_since_last_service": time_since_last_service,
        "avg_trip_duration": avg_trip_duration,
        "city_drive_ratio": city_drive_ratio
    }])

    prob_failure = model.predict_proba(input_df)[0][1]

    if prob_failure >= threshold:
        status = "RISK"
        msg = "⚠️ Engine is at higher risk of failure soon.\nRecommend inspection."
    else:
        status = "OK"
        msg = "✅ Engine is unlikely to fail in the near future."

    return prob_failure, status, msg


# ================
# 2. TKINTER GUI
# ================

def create_gui():
    root = tk.Tk()
    root.title("Car Engine Failure Predictor")
    root.geometry("480x520")
    root.resizable(False, False)

    mainframe = ttk.Frame(root, padding="15")
    mainframe.pack(fill="both", expand=True)

    title_label = ttk.Label(
        mainframe,
        text="Car Engine Failure Predictor",
        font=("Segoe UI", 16, "bold")
    )
    title_label.grid(row=0, column=0, columnspan=2, pady=(0, 10))

    desc_label = ttk.Label(
        mainframe,
        text="Enter current engine parameters and press Predict.",
        font=("Segoe UI", 9)
    )
    desc_label.grid(row=1, column=0, columnspan=2, pady=(0, 15))

    labels = [
        "Mileage (km):",
        "Engine Temp (°C):",
        "Oil Pressure (bar):",
        "RPM:",
        "Error Code Count:",
        "Time Since Last Service (days):",
        "Avg Trip Duration (min):",
        "City Drive Ratio (0–1):"
    ]

    default_values = [
        "150000",  # mileage
        "95",      # engine temp
        "2.8",     # oil pressure
        "2500",    # rpm
        "1",       # error_code_count
        "300",     # time_since_last_service
        "30",      # avg_trip_duration
        "0.6"      # city_drive_ratio
    ]

    entries = []

    for i, (label_text, default) in enumerate(zip(labels, default_values), start=2):
        lbl = ttk.Label(mainframe, text=label_text)
        lbl.grid(row=i, column=0, sticky="w", pady=3)

        entry = ttk.Entry(mainframe, width=20)
        entry.grid(row=i, column=1, sticky="w", pady=3)
        entry.insert(0, default)
        entries.append(entry)

    result_frame = ttk.LabelFrame(mainframe, text="Prediction Result", padding="10")
    result_frame.grid(row=11, column=0, columnspan=2, pady=15, sticky="we")

    prob_label_var = tk.StringVar(value="Probability of failure: -")
    status_label_var = tk.StringVar(value="Status: -")
    message_label_var = tk.StringVar(value="Message will appear here.")

    prob_label = ttk.Label(result_frame, textvariable=prob_label_var, font=("Segoe UI", 10, "bold"))
    prob_label.pack(anchor="w", pady=2)

    status_label = ttk.Label(result_frame, textvariable=status_label_var, font=("Segoe UI", 11, "bold"))
    status_label.pack(anchor="w", pady=2)

    message_label = ttk.Label(result_frame, textvariable=message_label_var, font=("Segoe UI", 10), wraplength=420)
    message_label.pack(anchor="w", pady=2)

    def on_predict():
        try:
            mileage = float(entries[0].get())
            engine_temp = float(entries[1].get())
            oil_pressure = float(entries[2].get())
            rpm = float(entries[3].get())
            error_code_count = float(entries[4].get())
            time_since_last_service = float(entries[5].get())
            avg_trip_duration = float(entries[6].get())
            city_drive_ratio = float(entries[7].get())

            if not (0.0 <= city_drive_ratio <= 1.0):
                raise ValueError("City drive ratio must be between 0 and 1")

            prob, status, msg = predict_engine_failure(
                mileage, engine_temp, oil_pressure, rpm,
                error_code_count, time_since_last_service,
                avg_trip_duration, city_drive_ratio
            )

            prob_label_var.set(f"Probability of failure: {prob:.2%}")
            status_label_var.set(f"Status: {status}")
            message_label_var.set(msg)

        except ValueError as e:
            messagebox.showerror("Input Error", f"Invalid input: {e}")

    predict_button = ttk.Button(
        mainframe,
        text="Predict",
        command=on_predict
    )
    predict_button.grid(row=12, column=0, columnspan=2, pady=10)

    note_label = ttk.Label(
        mainframe,
        text="*This is a demo using synthetic data.\nNot for real-world safety-critical use.",
        font=("Segoe UI", 8, "italic")
    )
    note_label.grid(row=13, column=0, columnspan=2, pady=(5, 0))

    root.mainloop()


if __name__ == "__main__":
    create_gui()


Generating synthetic engine data...
Training RandomForest model...
Model training complete.
